# TP EEG — Notebook 0 : préparer un jeu de données

**Objectif :** disposer d'un fichier de données EEG, que vous ayez ou non réussi à enregistrer avec le casque OpenBCI le jour du TP.

Ce notebook crée un fichier `eeg_data.csv` qui imite la sortie d'un casque **OpenBCI Cyton + Daisy** (16 canaux, 125 échantillons par seconde) pendant une expérience **yeux ouverts / yeux fermés**.

> 💡 Si vous avez un vrai enregistrement, sautez à la fin du notebook pour voir comment l'utiliser à la place.

---
### Comment exécuter une cellule ?
Cliquez dans la cellule de code grise, puis appuyez sur **Maj + Entrée**. Le résultat s'affiche juste en dessous.

## 1. Importer des outils

En Python, on ne réinvente pas tout : on **importe** des boîtes à outils (appelées *bibliothèques*). Ici :
- `numpy` : pour calculer avec des tableaux de nombres (on lui donne le surnom `np`).
- `pandas` : pour manipuler des tableaux type tableur (surnom `pd`).
- `matplotlib.pyplot` : pour dessiner des courbes (surnom `plt`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Bibliothèques importées avec succès !")

Bibliothèques importées avec succès !


## 2. Définir les paramètres de l'expérience

Une **variable** est une étiquette qui retient une valeur. On lit la ligne `fs = 125` comme « la variable *fs* vaut 125 ».

In [ ]:
fs = 125            # fréquence d'échantillonnage (échantillons par seconde)
n_canaux = 16        # nombre d'électrodes
duree_bloc = 30      # secondes pour chaque condition (yeux ouverts, puis fermés)

# Noms d'électrodes (système 10-20) typiques pour 16 voies
noms_canaux = ['Fp1','Fp2','F3','F4','C3','C4','P3','P4',
               'O1','O2','F7','F8','T3','T4','P7','P8']

print("Durée totale prévue :", 2 * duree_bloc, "secondes")

Durée totale prévue : 60 secondes


## 3. Fabriquer un signal EEG réaliste

Un signal EEG, c'est surtout un mélange d'ondes de différentes fréquences plus du bruit. Le phénomène vedette du TP : quand on **ferme les yeux**, le rythme **alpha** (~10 Hz) augmente fortement, surtout à l'arrière du crâne (électrodes O1, O2).

Vous n'avez pas besoin de tout comprendre dans cette cellule : elle sert juste à fabriquer les données. On y reviendra.

In [ ]:
rng = np.random.default_rng(42)  # graine aléatoire : résultats reproductibles

def bloc(condition, duree):
    """Crée un bloc de signal pour une condition ('ouvert' ou 'ferme')."""
    n = duree * fs
    t = np.arange(n) / fs                      # axe du temps en secondes
    signal = np.zeros((n, n_canaux))
    for c in range(n_canaux):
        # bruit de fond (toutes fréquences)
        x = rng.normal(0, 8, n)
        # rythme alpha 10 Hz : fort si yeux fermés ET électrode arrière
        arriere = noms_canaux[c].startswith(('O', 'P'))
        amp_alpha = 18 if (condition == 'ferme' and arriere) else 3
        x += amp_alpha * np.sin(2*np.pi*10*t + rng.uniform(0, 6))
        # un peu de rythme à 50 Hz : le bruit du secteur électrique
        x += 4 * np.sin(2*np.pi*50*t)
        signal[:, c] = x
    return signal

sig_ouvert = bloc('ouvert', duree_bloc)
sig_ferme  = bloc('ferme',  duree_bloc)
signal = np.vstack([sig_ouvert, sig_ferme])   # on empile les deux blocs

print("Forme du tableau de signal :", signal.shape, "(lignes = temps, colonnes = canaux)")

Forme du tableau de signal : (7500, 16) (lignes = temps, colonnes = canaux)


## 4. Ajouter une colonne « marqueur »

Pour savoir quelle partie correspond à *yeux ouverts* (0) et *yeux fermés* (1), on ajoute une colonne d'étiquettes.

In [ ]:
marqueur = np.concatenate([
    np.zeros(len(sig_ouvert)),   # 0 = yeux ouverts
    np.ones(len(sig_ferme))      # 1 = yeux fermés
])

print("Nombre d'échantillons yeux ouverts :", int((marqueur == 0).sum()))
print("Nombre d'échantillons yeux fermés  :", int((marqueur == 1).sum()))

Nombre d'échantillons yeux ouverts : 3750
Nombre d'échantillons yeux fermés  : 3750


## 5. Enregistrer dans un fichier CSV

On range tout dans un tableau `pandas` (un *DataFrame*) puis on l'écrit sur le disque. C'est ce fichier que les prochains notebooks liront.

In [ ]:
df = pd.DataFrame(signal, columns=noms_canaux)
df['marqueur'] = marqueur
df.to_csv('eeg_data.csv', index=False)

print("Fichier 'eeg_data.csv' enregistré.")
df.head()   # aperçu des 5 premières lignes

Fichier 'eeg_data.csv' enregistré.


,Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,F7,F8,T3,T4,P7,P8,marqueur
0,3.149829,-2.709558,-1.492324,9.691624,6.062952,-14.087615,8.590432,11.357484,-21.735649,-0.368653,2.142271,3.937667,3.018341,5.288791,-0.165667,-7.801041,0.0
1,-3.940764,2.080968,4.768776,6.975758,6.424492,6.321382,4.121725,-1.628167,-13.953998,-0.037136,7.183373,7.732842,11.185640,0.941258,-4.197788,19.802015,0.0
2,5.041535,1.934637,0.096604,2.186479,-0.056431,6.145486,-11.732975,-9.617510,-6.911511,-1.248521,-0.847047,3.007545,4.144338,-10.367313,-8.762843,-2.817406,0.0
3,14.281968,0.264665,-1.257113,-7.481322,6.607815,-4.121077,-19.261192,13.198531,-3.267288,4.745715,2.561197,6.650958,8.608170,6.089910,15.800501,-2.089374,0.0
4,-15.625713,-9.016426,-17.796552,-2.518915,-15.824944,8.372752,15.697767,-9.755227,-13.445890,-12.452746,3.157533,6.997422,-3.012202,-0.502373,6.581649,0.614163,0.0


## 6. Et avec un vrai enregistrement OpenBCI ?

Le GUI OpenBCI exporte un fichier `.txt`/`.csv` avec des lignes de commentaire en tête (commençant par `%`) puis les colonnes de canaux. Pour le charger :

```python
df = pd.read_csv('mon_enregistrement.txt', comment='%')
# Les colonnes utiles sont en général 'EXG Channel 0' à 'EXG Channel 15'
```

L'important pour la suite : obtenir un tableau `n_échantillons × 16 canaux`. Gardez bien en tête que **fs = 125 Hz** avec le Daisy branché.